In [79]:
# TODO
# Make notebook runable
# Update paths
# Update input data
# Delete IFFs and nurse estimates

In [80]:
import pandas as pd
import numpy as np
import pickle

from tjn_tools.style_guide import *
from tjn_tools.data_processing import *
import tjn_tools
from config_new import YEAR_CBCR, YEAR_SOTJ, UNILATERAL_CROSS, UNILATERAL_PANEL, BILATERAL_CROSS

Note: Originally this was named 98.combine_parts.ipynb, but I renamed it to 98.combine_parts.ipynb to make it run last.

In [81]:
# Defines file paths for different directories related to estimations data
path_files = "../../data/raw/estimations/"
path_files_final = "../../data/final/estimations/"
path_files_final_analysis = "../../data/final/analysis/"
path_files_temp = "../../data/intermediate/estimations/"
path_figures = "../../data/final/estimations/figures/"

In [82]:
# TODO Transfer these paths to config file
# Define file path for offshore wealth tax evasion data
tax_evasion_file_output = f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Data/Projects/2006 Offshore wealth/Tables/Offshore wealth Table Full results raw.xlsx"

In [83]:
# Define info expenditures file path and columns of interest to be used in the analysis
info_expenditures_output = f"{path_files_final}{YEAR_CBCR}_info_expenditures_new.csv"
cols_other_info = ['who_gvt_health_expenditure','govt_exp_educ_gdp',
        'total_taxes_revenue', 'cit_revenue', 'iso3', 'gdp', 'population',
        'region_tjn',"ukt","oecd","oecd_oct","g20","eu28","month_wage","fsi_2022_rank","fsi_2022_score","cthi_2021_rank","cthi_2021_share","cthi_2021_score"]


In [84]:
# Read cbcr etr rates data
etr_output = f"{path_files_final}{YEAR_CBCR}_cbcr_etr_rates_new.xlsx"
df_etrs = pd.read_excel(etr_output,index_col=0)
# Create dictionary with iso3 as key and etr as value
iso3_to_etr = df_etrs["ETR_total"].to_dict()

In [85]:
# Define tax avoidance sotj tavle file path to be used in the analysis
file_output = f"{YEAR_CBCR}_tax_avoidance_sotj_table_new_additionalcountries.xlsx"
sotj_table_output = f"{path_files_final_analysis}{file_output}"

In [86]:
# Read series iso3 to corporate income tax (cit) dictionnary
iso3_to_cit_output = f"{path_files_final}{YEAR_CBCR}_iso3_to_cit_new.dump"
iso3_to_cit = pickle.load(open(iso3_to_cit_output,"rb+"))

In [87]:
# Define file paths for combined output. One for saving the output locally and another in SharePoint
combined_table = f"{YEAR_CBCR}combined_output_additionalcountries.xlsx"
workstream_path = f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/{YEAR_SOTJ} Report/Combined_offshore_corporate/"
final_table_output = f"{path_files_final_analysis}{combined_table}"
final_table_output_workstream = f"{workstream_path}{combined_table}"

# 1. Read data

In [88]:
# Read cthi unilateral cross data for 2021 - TODO Update with latest data
# Note:Some variables were not added before: "EU28 OECT", "EU27", "EU27 OCT", "GBR OCT", th_eu_blacklist_201006, th_eu_greylist_201006, th_unctad2015
countryYearBase = pd.read_stata(f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/GSW/210211 countryYearBase for CTHI2021.dta")
# Select columns of interest
countryYearBase = countryYearBase.loc[:,["country","EU27","EU27_OCT","EU28_OCT","GBR_OCT","th_eu_blacklist_201006", "th_eu_greylist_201006", "th_unctad2015"]]
# Generate iso3 column
countryYearBase["iso3"] = countryYearBase["country"].apply(get_iso3, print_failure = False)
# Clean dataframe
countryYearBase = countryYearBase.loc[countryYearBase["country"] != "West Bank and Gaza"]
countryYearBase = countryYearBase.drop(columns=["country"])
countryYearBase = countryYearBase.drop_duplicates()
countryYearBase.sample(10)

,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,iso3
13150,0.0,0.0,0.0,0.0,1.0,0.0,1.0,PAN
12920,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PAK
6916,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GLP
7372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,GNB
3800,0.0,0.0,0.0,0.0,0.0,0.0,0.0,COM
8132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,IRQ
608,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ATA
11552,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MMR
6840,0.0,0.0,0.0,0.0,0.0,0.0,1.0,GRD
3876,0.0,0.0,0.0,0.0,0.0,0.0,0.0,COD


In [89]:
# Read info expenditures data and merge it with countryYearBase
other_info = pd.read_csv(info_expenditures_output, sep="\t", usecols=cols_other_info)
other_info = pd.merge(other_info,countryYearBase,how="left")
other_info.head()

,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015
0,ABW,6.458101e+08,NaN,NaN,105962.0,1986.169974,NaN,3.202235e+09,56.0,0.002126,70.134565,75.0,70.925,Caribbean/American isl.,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,AFG,1.689714e+09,NaN,5.893645e+08,36686784.0,82.518273,8.951096e+07,1.841885e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AGO,1.514723e+10,NaN,1.590632e+09,31273533.0,291.116457,8.314383e+08,7.779294e+10,NaN,NaN,NaN,33.0,79.450,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.005760,100.000000,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
4,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
# Read Unilateral cross data with selected columns
cols = {"iso3":"iso3","ps_jansky19": "TA: JP 2019 (USD million)"} #Update this list for all useful comparisons (TWZ, C&J)
# ext_estimates = pd.read_csv(UNILATERAL_CROSS,skiprows=1,sep="\t",usecols=(list(cols.keys())))
ext_estimates = pd.read_csv(UNILATERAL_CROSS,usecols=(list(cols.keys())))
# cols = [_ for _ in ohter_m.columns if ("ps_" in _) and not ("_so") in _ and (_ not in ("ps_cobham2018","ps_torslov2018") )]
# Filter by dropñing rows with least two nan values and remove duplicates
ext_estimates = ext_estimates[list(cols)].dropna(thresh=2).drop_duplicates(subset=["iso3"])
# Rename columns
ext_estimates = ext_estimates.rename(columns=cols)
# Convert values from USD million to USD except for the first column
ext_estimates[list(ext_estimates.columns )[1:]] /= 1E6

In [91]:
# Merge other info with ext_estimates
other_info = pd.merge(other_info,ext_estimates,how="left",validate="1:1")
other_info.head()

,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million)
0,ABW,6.458101e+08,NaN,NaN,105962.0,1986.169974,NaN,3.202235e+09,56.0,0.002126,70.134565,75.0,70.925,Caribbean/American isl.,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,NaN
1,AFG,1.689714e+09,NaN,5.893645e+08,36686784.0,82.518273,8.951096e+07,1.841885e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,AGO,1.514723e+10,NaN,1.590632e+09,31273533.0,291.116457,8.314383e+08,7.779294e+10,NaN,NaN,NaN,33.0,79.450,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.005760,100.000000,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,NaN
4,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [92]:
#Income class
# Read Unilateral panel data with selected columns
income_class = pd.read_csv(UNILATERAL_PANEL,usecols=["iso3","year","gdp","population"])
# Calculate GDP per capita
income_class["GDPpc"] = income_class["gdp"]/income_class["population"]
# Calculate income class
income_class["income_class"] = pd.cut(income_class["GDPpc"],[0,1025,3995,12375,np.inf],labels=["L","LM","UM","H"]) #These thresholds are for 2018 (from analytical classifications here: http://databank.worldbank.org/data/download/site-content/OGHIST.xls)
# Identify the ISO3 codes for which the "IncomeClass" information is missing by calculating the set difference between all ISO3 codes and the ISO3 codes with available "IncomeClass" information
missing_income = set(income_class["iso3"]) - set(income_class.dropna(subset=["income_class"])["iso3"])
print(missing_income)
# Separate the rows from the "income_class" DataFrame into two subsets based on the presence or absence of "IncomeClass" information, dropping any duplicate rows and renaming columns in one of the subsets
info_on_income_class = income_class.loc[~income_class["iso3"].isin(missing_income)].dropna(subset=["income_class"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = income_class.loc[income_class["iso3"].isin(missing_income)].dropna(subset=["income_class"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = no_info_on_income_class.rename(columns = {"income_class": "income_class_na","income_class": "income_class"})
# Combine the "info_on_income_class" and "no_info_on_income_class" DataFrames into a single DataFrame named "income_class" and replaces the values in the "IncomeClass" column with descriptive labels.
income_class = pd.concat([info_on_income_class, no_info_on_income_class])
income_class["income_class"] = income_class["income_class"].map({'H':"High income", 'L':"Low income", 'LM':"Lower-middle income", 'UM':"Upper-middle income"})
# Keep columns of interest
income_class = income_class.loc[:,["iso3","income_class"]]
# Merge other info with income_class
other_info = pd.merge(other_info,income_class,how="left",validate="1:1")
other_info.head()

{'SGS', 'ATF', 'AIA', 'VAT', 'JEY', 'NIU', 'BLM', 'REU', 'IOT', 'COK', 'MSR', 'BES', 'WLF', 'FLK', 'TWN', 'UMI', 'TMP', 'MTQ', 'ANT', 'HMD', 'MYT', 'XXK', 'PCN', 'WSH', 'ALA', 'CCK', 'SJM', 'SPM', 'GLP', 'BVT', 'GUF', 'ESH', 'SHN', 'NFK', 'PAL', 'GGY', 'CXR', 'TKL'}


,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million),income_class
0,ABW,6.458101e+08,NaN,NaN,105962.0,1986.169974,NaN,3.202235e+09,56.0,0.002126,70.134565,75.0,70.925,Caribbean/American isl.,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,NaN,High income
1,AFG,1.689714e+09,NaN,5.893645e+08,36686784.0,82.518273,8.951096e+07,1.841885e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Low income
2,AGO,1.514723e+10,NaN,1.590632e+09,31273533.0,291.116457,8.314383e+08,7.779294e+10,NaN,NaN,NaN,33.0,79.450,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Lower-middle income
3,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.005760,100.000000,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN
4,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
#Tax avoidance
# Read an Excel file into the "tax_avoidance_file" DataFrame, adjusts the values in the "CIT" and "ETR" columns by dividing them by 100, maps country names to ISO-3 codes, and replaces a specific code in the "ISO-3 code of country" column
tax_avoidance_file = pd.read_excel(sotj_table_output).drop_duplicates()
tax_avoidance_file["CIT"] /= 100
tax_avoidance_file["ETR"] /= 100
tax_avoidance_file["Name"].loc[tax_avoidance_file["Name"] == "Cura�ao"] = "Curacao"
tax_avoidance_file["ISO-3 code of country"] = tax_avoidance_file["Name"].map(get_iso3).replace("SCG","SRB") 

tax_avoidance_file.head()

 Africa not matched to any file
 Asia not matched to any file
 Caribbean/American isl. not matched to any file
 Europe not matched to any file
 Latin America not matched to any file
 Northern America not matched to any file
 Oceania not matched to any file


,Name,MNCs,CIT,ETR,Profit loss (M),Min. Profit loss (M),Max. Profit loss (M),Revenue loss using CIT (M),Min. Revenue loss using CIT (M),Max. Revenue loss using CIT (M),Revenue loss using ETR (M),Min Revenue loss using ETR (M),Max. Revenue loss using ETR (M),Robust,3+reporters,N_reporters,gdp,pop,Profit loss per gdp (%),Profit loss per health (%),Profit loss per educ (%),Profit loss per tax_revenue (%),Profit loss per pop ($ per capita),ISO-3 code of country
0,Africa,Foreign (in)Foreign (out)Foreign (in)Foreign (...,0.2787,0.2917,17766,15263,20610,5168.0,4474.4,5960.0,4624.4,4081.1,5276.9,121,56,745,5241974361408,2524099480,0.3,23.2,5.8,2.0,7,NaN
1,Algeria,Foreign (in),0.2600,0.5117,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,14,174910878623,41927007,0.0,0.0,0.0,0.0,0,DZA
2,Algeria,Foreign (out),0.2600,0.5117,33,30,35,8.5,7.8,9.2,16.7,15.4,18.1,2,1,14,174910878623,41927007,0.0,0.5,0.3,0.0,1,DZA
3,Angola,Foreign (in),0.3000,0.2884,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,8,77792944472,31273533,0.0,0.0,0.0,0.0,0,AGO
4,Angola,Foreign (out),0.3000,0.2884,420,363,492,125.9,108.9,147.5,121.0,104.7,141.9,2,1,8,77792944472,31273533,0.5,50.5,26.4,2.8,13,AGO


In [94]:
#Tax evasion
# The code reads an Excel file into the "tax_evasion_file" DataFrame, maps country names to ISO-3 codes, removes the "Country" column, and displays the initial rows of the updated DataFrame.
tax_evasion_file = pd.read_excel(tax_evasion_file_output)
tax_evasion_file["ISO-3 code of country"] = tax_evasion_file["Country"].map(get_iso3)
tax_evasion_file = tax_evasion_file.drop(columns=["Country"])
tax_evasion_file.head()


,Share of global offshore wealth owned by citizens of country,Offshore wealth owned by citizens of country (USD billion),Offshore wealth owned by citizens of country (% of GDP),Tax revenue loss: Offshore wealth (USD million),Share of global tax loss inflicted by country,Tax loss inflicted on other countries,ISO-3 code of country
0,0.209388,2081.789062,0.097129,38513.097656,0.121643,20859.845703,USA
1,0.122834,1221.248779,0.431413,27478.097656,0.178115,30543.751953,GBR
2,0.055693,553.710754,1.389173,13289.058594,0.052210,8953.128906,IRL
3,0.045214,449.527100,0.116424,10114.359375,0.000000,0.000000,DEU
4,0.045205,449.440491,0.031474,10112.411133,0.000000,0.000000,CHN


In [95]:
#other_info.loc[other_info["iso3"].isin([get_iso3(_) for _ in ["Malaysia","Vietnam","Thailand","Cambodia","Indonesia","Myanmar","Philippines"]]),["iso3","month_wage"]]

In [96]:
# Select specific columns from the "tax_avoidance_file" DataFrame where the "MNCs" column is equal to "Domestic", "Foreign (out)", or "Foreign (in)", and assigns the resulting DataFrame to "ta_dom", "ta_fo", and "ta_ga" respectively. It then renames the columns in each DataFrame to include corresponding suffixes indicating the type of MNCs
ta_dom = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Domestic",["ISO-3 code of country","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"N_reporters"]]   
ta_dom.columns = list(ta_dom.columns[:1]) + [_ + " - dom" for _ in ta_dom.columns[1:]]
ta_fo = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (out)",["ISO-3 code of country","ETR","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust","N_reporters"]]
ta_fo.columns = list(ta_fo.columns[:2]) + [_ + " - for lose" for _ in ta_fo.columns[2:]]  # no Foreign (out) for JER and GGY
ta_ga = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (in)",["ISO-3 code of country","CIT","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust"]]
ta_ga.columns = list(ta_ga.columns[:2]) + [_ + " - for gain" for _ in ta_ga.columns[2:]]

# Concatenate the "ta_dom", "ta_fo", and "ta_ga" DataFrames along the column axis, with the ISO-3 code of country as the common index, and assigns the result to the "ta" DataFrame
ta = pd.concat([ta_dom.set_index("ISO-3 code of country"),ta_fo.set_index("ISO-3 code of country"),ta_ga.set_index("ISO-3 code of country")],axis=1,sort=False)
ta = ta.reset_index().rename(columns={"index":"ISO-3 code of country"})

# Calculate revenue loss values based on profit loss and tax rates for different types of MNCs and assigns them to the "ta" DataFrame
for tax in ["CIT","ETR"]:
    for var in ["Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)']:
        for end in ["- dom","- for lose","- for gain"]:
            ta[f"Revenue loss using {tax} (M) {end}"] = ta[f"{var} {end}"]*ta[tax]

ta = ta.rename(columns={"N_reporters - dom":"Reporting country"})
ta.head()

,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom
0,ZAF,842.0,778.0,902.0,1.0,0.1795,4121.0,3912.0,4309.0,1206.5200,773.4655,2.0,22.0,0.2800,0.0,0.0,0.0,0.00,0.000,1.0,252.5600,161.9090
1,HKG,-19884.0,-19908.0,-19860.0,1.0,0.0645,1382.0,1330.0,1438.0,237.2700,92.7510,2.0,25.0,0.1650,-35898.0,-37358.0,-34642.0,-5715.93,-2234.409,2.0,-3276.9000,-1280.9700
2,IND,34685.0,34330.0,35049.0,1.0,0.4132,30553.0,29268.0,31962.0,15444.0384,13206.6984,2.0,22.0,0.4832,0.0,0.0,0.0,0.00,0.000,1.0,16935.6768,14482.2468
3,IDN,-2211.0,-2251.0,-2173.0,1.0,0.2701,10946.0,10286.0,11715.0,2928.7500,3164.2215,2.0,20.0,0.2500,0.0,0.0,0.0,0.00,0.000,1.0,-543.2500,-586.9273
4,JPN,10874.0,9966.0,11780.0,1.0,0.2815,516.0,499.0,534.0,158.8116,150.3210,2.0,21.0,0.2974,0.0,0.0,0.0,0.00,0.000,1.0,3503.3720,3316.0700


In [97]:
def return_gain(common_var = "Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1):
    """ Calculate the sum of averaged values from the "ta" DataFrame based on the given parameters, printing intermediate results, and returns the resulting values"""
    vals = ((ta["{}{}".format(common_var,vars_[0])]+sign*ta["{}{}".format(common_var,vars_[0])].abs())/2).fillna(0)
    print(vals)
    for v in vars_[1:]:
        vals += ((ta["{}{}".format(common_var,v)]+sign*ta["{}{}".format(common_var,v)].abs())/2).fillna(0)
    print(vals)
    return vals

# Iterate over variations "dom", "for lose", and "for gain", and for each variation, it iterates over states "" (empty string), "Min. ", and "Max. ". It fills missing values in columns of the "ta" DataFrame with names in the format "{st}Profit loss (M) - {v}" (where "{st}" represents the state and "{v}" represents the variation) with 0.
for v in ["dom","for lose","for gain"]:
    for st in ["","Min. ","Max. "]:
        ta[f"{st}Profit loss (M) - {v}"] = ta[f"{st}Profit loss (M) - {v}"].fillna(0)
# Maps the values in the "ISO-3 code of country" column of the "ta" DataFrame to corresponding values from the "iso3_to_etr" and "iso3_to_cit" dictionaries, and assigns the mapped values to the "ETR" and "CIT" columns of the "ta" DataFrame, respectively.
ta["ETR"] = ta["ISO-3 code of country"].map(iso3_to_etr)
ta["CIT"] = ta["ISO-3 code of country"].map(iso3_to_cit)
# Calculate and assigns the summed profit gain and profit loss values based on different variations and states to the corresponding columns in the "ta" DataFrame
for st in ["","Min. ","Max. "]:
    ta[f"{st}Profit gain (M)"] = -return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1)
    ta[f"{st}Profit loss (M)"] = return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=1)
# Calculate and assigns the revenue gain and revenue loss values based on different variations and states using the "CIT" and "ETR" columns in the "ta" DataFrame multiplied by the corresponding profit gain and profit loss values, and assigns the calculated values to the corresponding columns in the "ta" DataFrame
for st in ["","Min. ","Max. "]:
    ta[f"{st}Revenue gain using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue gain using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue loss using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit loss (M)"]
    ta[f"{st}Revenue loss using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit loss (M)"]

# Assign the sum of the "Reporting country" column and a binary indicator (1 if "N_reporters - for lose" is greater than 3, 0 otherwise) to the "Robust" column in the "ta" DataFrame
ta["Robust"] = ta["Reporting country"]+((ta["N_reporters - for lose"])>3).astype(int)

ta.sort_values(by="Profit gain (M)").tail(20)

0          0.0
1     -19884.0
2          0.0
3      -2211.0
4          0.0
        ...   
208        0.0
209        0.0
210        0.0
211        0.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0          0.0
1     -55782.0
2          0.0
3      -2211.0
4          0.0
        ...   
208     -224.0
209        0.0
210        0.0
211     -833.0
212      -20.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0        842.0
1          0.0
2      34685.0
3          0.0
4      10874.0
        ...   
208        0.0
209        0.0
210        0.0
211        0.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0       4963.0
1       1382.0
2      65238.0
3      10946.0
4      11390.0
        ...   
208        0.0
209        0.0
210        0.0
211        0.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0          0.0
1     -19908.0
2          0.0
3      -2251.0
4          0.0
        ...   
208        0.0
209    

,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,Profit gain (M),Profit loss (M),Min. Profit gain (M),Min. Profit loss (M),Max. Profit gain (M),Max. Profit loss (M),Revenue gain using CIT (M),Revenue gain using ETR (M),Revenue loss using CIT (M),Revenue loss using ETR (M),Min. Revenue gain using CIT (M),Min. Revenue gain using ETR (M),Min. Revenue loss using CIT (M),Min. Revenue loss using ETR (M),Max. Revenue gain using CIT (M),Max. Revenue gain using ETR (M),Max. Revenue loss using CIT (M),Max. Revenue loss using ETR (M),Robust
21,IMN,-395.0,-395.0,-394.0,1.0,0.076711,240.0,225.0,266.0,0.0000,20.4022,2.0,9.0,0.00000,-13025.0,-14357.0,-12222.0,-0.0000,-937.4274,2.0,-0.0000,-30.2198,13420.0,240.0,14752.0,225.0,12616.0,266.0,0.000000,1029.459777,0.000000,18.410607,0.000000,1131.638646,0.000000,17.259944,0.000000,967.784243,0.000000,20.405089,2.0
5,MYS,-16669.0,-16739.0,-16604.0,1.0,0.218637,3229.0,3066.0,3418.0,820.3200,747.1748,2.0,21.0,0.24000,0.0,0.0,0.0,0.0000,0.0000,1.0,-3984.9600,-3629.6344,16669.0,3229.0,16739.0,3066.0,16604.0,3418.0,4000.559911,3644.459591,774.959983,705.978764,4017.359910,3659.764178,735.839984,670.340939,3984.959911,3630.248188,820.319982,747.301151,2.0
27,NOR,-18307.0,-18345.0,-18270.0,1.0,0.332999,4209.0,3919.0,4515.0,1038.4500,1503.4950,2.0,15.0,0.23000,0.0,0.0,0.0,0.0000,0.0000,1.0,-4202.1000,-6083.9100,18307.0,4209.0,18345.0,3919.0,18270.0,4515.0,4210.610076,6096.215837,968.070018,1401.593514,4219.350077,6108.869806,901.370016,1305.023754,4202.100076,6083.894868,1038.450019,1503.491260,2.0
152,JEY,0.0,0.0,0.0,NaN,0.007979,1432.0,1225.0,1837.0,0.0000,14.6960,2.0,11.0,0.00000,-18727.0,-23993.0,-16067.0,-0.0000,-128.5360,2.0,NaN,NaN,18727.0,1432.0,23993.0,1225.0,16067.0,1837.0,0.000000,149.420903,0.000000,11.425788,0.000000,191.437802,0.000000,9.774155,0.000000,128.197023,0.000000,14.657243,NaN
128,VGB,0.0,0.0,0.0,NaN,0.004546,1408.0,1330.0,1535.0,0.0000,6.9075,2.0,22.0,0.00000,-21367.0,-23211.0,-20238.0,-0.0000,-91.0710,2.0,NaN,NaN,21367.0,1408.0,23211.0,1330.0,20238.0,1535.0,0.000000,97.125658,0.000000,6.400193,0.000000,105.507730,0.000000,6.045637,0.000000,91.993685,0.000000,6.977483,NaN
38,MEX,-21573.0,-21706.0,-21448.0,1.0,0.200267,19859.0,19407.0,20408.0,6122.4000,4087.7224,2.0,22.0,0.30000,0.0,0.0,0.0,0.0000,0.0000,1.0,-6434.4000,-4296.0344,21573.0,19859.0,21706.0,19407.0,21448.0,20408.0,6471.900257,4320.355998,5957.700237,3977.098677,6511.800259,4346.991485,5822.100231,3886.578077,6434.400256,4295.322646,6122.400243,4087.045159,2.0
135,PRI,0.0,0.0,0.0,NaN,0.027705,484.0,450.0,525.0,204.7500,14.5425,2.0,13.0,0.39000,-23086.0,-25057.0,-21550.0,-8404.5000,-596.9350,2.0,NaN,NaN,23086.0,484.0,25057.0,450.0,21550.0,525.0,9003.540000,639.597299,188.760000,13.409213,9772.230000,694.203826,175.500000,12.467244,8404.500000,597.042441,204.750000,14.545117,NaN
149,GIB,0.0,0.0,0.0,NaN,0.009928,93.0,76.0,115.0,0.0000,1.1385,2.0,7.0,0.00000,-26789.0,-33063.0,-21797.0,-0.0000,-215.7903,2.0,NaN,NaN,26789.0,93.0,33063.0,76.0,21797.0,115.0,0.000000,265.972365,0.000000,0.923343,0.000000,328.263254,0.000000,0.754560,0.000000,216.409707,0.000000,1.141768,NaN
8,KOR,-30173.0,-30417.0,-29936.0,1.0,0.300256,963.0,920.0,1007.0,276.9250,302.4021,2.0,21.0,0.27500,0.0,0.0,0.0,0.0000,0.0000,1.0,-8232.4000,-8989.7808,30173.0,963.0,30417.0,920.0,29936.0,1007.0,8297.575180,9059.618560,264.825006,289.146345,8364.675181,9132.880978,253.000005,276.235345,8232.400178,8988.457933,276.925006,302.357601,2.0

In [98]:
# Select the rows in the "ta" DataFrame where the "ISO-3 code of country" is equal to "USA" and retrieves the columns containing "Revenue loss" and "CIT" in their names (excluding columns with decimal points) for those rows
ta.loc[ta["ISO-3 code of country"]=="USA",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

,Revenue loss using CIT (M) - for lose,Revenue loss using CIT (M) - for gain,Revenue loss using CIT (M) - dom,Revenue loss using CIT (M)
42,12014.19,0.0,129375.36,139815.99


In [99]:
# Select the rows in the "ta" DataFrame where the "ISO-3 code of country" is equal to "TCD" and retrieves the columns containing "Revenue loss" and "CIT" in their names (excluding columns with decimal points) for those rows
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

,Revenue loss using CIT (M) - for lose,Revenue loss using CIT (M) - for gain,Revenue loss using CIT (M) - dom,Revenue loss using CIT (M)
52,0.7,-13.3,NaN,0.35


In [100]:
# Select the rows in the "ta" DataFrame where the "ISO-3 code of country" is equal to "TCD" and retrieves the columns containing "Profit loss" in their names (excluding columns with decimal points) for those rows.
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Profit loss" in _) and ("." not in _) ]]

,Profit loss (M) - dom,Profit loss (M) - for lose,Profit loss (M) - for gain,Profit loss (M)
52,0.0,1.0,-48.0,1.0


In [101]:
## MERGE FILES
# Merge the "tax_evasion_file" DataFrame with the "ta" DataFrame using an outer join, and then merges the resulting DataFrame with the "other_info" DataFrame based on the "ISO-3 code of country" column, using a left join. The resulting DataFrame is assigned to the variable "df_merged"
df_merged = pd.merge(tax_evasion_file,ta,how="outer").dropna(subset=["ISO-3 code of country"])
# df_merged = pd.merge(df_merged,iff_file,how="outer").dropna(subset=["ISO-3 code of country"]) #TODO delete this line and nurses-wise one
# df_merged = pd.merge(df_merged,childrens_file,how="left").dropna(subset=["ISO-3 code of country"])
df_merged = pd.merge(df_merged,other_info,left_on="ISO-3 code of country",right_on="iso3",how="left")

df_merged["Offshore wealth owned by citizens of country (USD billion)"] *= 1000*0.05 #Convert to million and multiply by the rate of return
#df_merged["fsi_2022_share"] *= 100
#df_merged["cthi_2021_share"] *= 100

# Map the "CIT" values in the "df_merged" DataFrame based on the "ISO-3 code of country" using the "iso3_to_cit" dictionary.
df_merged["CIT"] = df_merged["ISO-3 code of country"].map(iso3_to_cit)
df_merged.head()


,Share of global offshore wealth owned by citizens of country,Offshore wealth owned by citizens of country (USD billion),Offshore wealth owned by citizens of country (% of GDP),Tax revenue loss: Offshore wealth (USD million),Share of global tax loss inflicted by country,Tax loss inflicted on other countries,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,Profit gain (M),Profit loss (M),Min. Profit gain (M),Min. Profit loss (M),Max. Profit gain (M),Max. Profit loss (M),Revenue gain using CIT (M),Revenue gain using ETR (M),Revenue loss using CIT (M),Revenue loss using ETR (M),Min. Revenue gain using CIT (M),Min. Revenue gain using ETR (M),Min. Revenue loss using CIT (M),Min. Revenue loss using ETR (M),Max. Revenue gain using CIT (M),Max. Revenue gain using ETR (M),Max. Revenue loss using CIT (M),Max. Revenue loss using ETR (M),Robust,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million),income_class
0,0.209388,104089.453125,0.097129,38513.097656,0.121643,20859.845703,USA,474005.0,469128.0,479168.0,1.0,0.227980,43832.0,43198.0,44497.0,12014.1900,10145.3160,2.0,27.0,0.27000,0.0,0.0,0.0,0.00,0.0000,1.0,129375.3600,109250.3040,-0.0,517837.0,-0.0,512326.0,-0.0,523665.0,-0.000000,-0.000000,139815.990000,118056.222832,-0.000000,-0.000000,138328.020000,116799.827781,-0.000000,-0.000000,141389.550000,119384.887386,2.0,USA,3.766917e+12,2.049875e+11,1.008652e+12,3.268382e+08,4400.00,1.747848e+12,2.053306e+13,25.0,0.011610,46.898267,1.0,67.425,Northern America,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.080189,High income
1,0.122834,61062.438965,0.431413,27478.097656,0.178115,30543.751953,GBR,70516.0,69887.0,71185.0,1.0,0.082819,17369.0,17044.0,17695.0,3362.0500,1465.1460,2.0,26.0,0.19000,-118088.0,-119520.0,-116726.0,-22177.94,-9664.9128,2.0,13525.1500,5894.1180,118088.0,87885.0,119520.0,86931.0,116726.0,88880.0,22436.719718,9779.973428,16698.149790,7278.580082,22708.799715,9898.570761,16516.889793,7199.570405,22177.939722,9667.173450,16887.199788,7360.985352,2.0,GBR,7.662662e+11,7.564826e+10,1.487426e+11,6.646034e+07,3216.68,2.261647e+11,2.878152e+12,13.0,0.031230,69.204656,13.0,47.175,Europe,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.013602,High income
2,0.055693,27685.537720,1.389173,13289.058594,0.052210,8953.128906,IRL,3435.0,3407.0,3465.0,1.0,0.115658,1469.0,1406.0,1538.0,192.2500,177.9466,2.0,19.0,0.12500,-40232.0,-42043.0,-38600.0,-4825.00,-4466.0200,2.0,433.1250,400.9005,40232.0,4904.0,42043.0,4813.0,38600.0,5003.0,5029.000000,4653.153367,613.000000,567.186919,5255.375000,4862.610037,601.625000,556.662039,4825.000000,4464.399482,625.375000,578.637062,2.0,IRL,7.161200e+10,1.225976e+10,1.308754e+10,4.867316e+06,3887.48,1.974858e+10,3.857367e+11,11.0,0.032960,77.079455,27.0,47.200,Europe,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,High income
3,0.045214,22476.354980,0.116424,10114.359375,0.000000,0.000000,DEU,24268.0,23740.0,24816.0,1.0,0.189484,30095.0,29539.0,30753.0,9173.6199,5827.6935,2.0,24.0,0.29825,0.0,0.0,0.0,0.00,0.0000,1.0,7402.6128,4702.6320,-0.0,54363.0,-0.0,53279.0,-0.0,55569.0,-0.000000,-0.000000,16213.765813,10300.930791,-0.000000,-0.000000,15890.462792,10095.529894,-0.000000,-0.000000,16573.455

In [102]:
tax_avoidance_file.loc[tax_avoidance_file["ISO-3 code of country"]=="TCD"]

,Name,MNCs,CIT,ETR,Profit loss (M),Min. Profit loss (M),Max. Profit loss (M),Revenue loss using CIT (M),Min. Revenue loss using CIT (M),Max. Revenue loss using CIT (M),Revenue loss using ETR (M),Min Revenue loss using ETR (M),Max. Revenue loss using ETR (M),Robust,3+reporters,N_reporters,gdp,pop,Profit loss per gdp (%),Profit loss per health (%),Profit loss per educ (%),Profit loss per tax_revenue (%),Profit loss per pop ($ per capita),ISO-3 code of country
15,Chad,Foreign (in),0.35,0.0,-48,-63,-38,-16.8,-22.1,-13.4,0.0,0.0,0.0,1,0,2,11239167048,15604210,-0.4,-61.4,-18.9,-6.0,-3,TCD
16,Chad,Foreign (out),0.35,0.0,1,1,2,0.5,0.4,0.7,0.0,0.0,0.0,1,0,2,11239167048,15604210,0.0,1.8,0.6,0.2,0,TCD


In [103]:
# Create a dictionary called "rename_cols" and assigns new column names to the respective keys based on the original column names in the DataFrame. The new column names are formatted to represent specific tax-related metrics in USD million.
rename_cols = {}
for st in ["","Min. ","Max. "]:
    rename_cols[f"{st}Profit gain (M)"] = f"TA: {st}Tax base gain (USD million)"
    rename_cols[f"{st}Profit loss (M)"] = f"TA: {st}Tax base loss (USD million)"
    rename_cols[f"{st}Revenue gain using CIT (M)"] = f"TA: {st}Tax revenue gain using CIT (USD million)"
    rename_cols[f"{st}Revenue gain using ETR (M)"] = f"TA: {st}Tax revenue gain using ETR (USD million)"
    rename_cols[f"{st}Revenue loss using CIT (M)"] = f"TA: {st}Tax revenue loss using CIT (USD million)"
    rename_cols[f"{st}Revenue loss using ETR (M)"] = f"TA: {st}Tax revenue loss using ETR (USD million)"
    
# Rename columns
rename_cols.update({"Offshore wealth owned by citizens of country (USD billion)": 'OW: Tax base loss (USD million)',
 "Tax revenue loss: Offshore wealth (USD million)": 'OW: Tax revenue loss (USD million)',
 'Share of global tax loss inflicted by country': 'Harm OW: Total (% total)',
 'Tax loss inflicted on other countries': 'Harm OW: Total (USD million)',
 'Reporting country': 'TA: Reporting country',
 'Robust': 'TA: Robust',
 'N_reporters - for lose': 'TA: Number countries reporting',
 'Outward Banking Positions': 'IFF: Outward Banking Positions',
 'Inward Banking Positions': 'IFF: Inward Banking Positions',
 'Outward FDI': 'IFF: Outward FDI',
 'Inward FDI': 'IFF: Inward FDI',
 'Outward Portfolio Inv.': 'IFF: Outward Portfolio Inv.',
 'Inward Portfolio Inv.': 'IFF: Inward Portfolio Inv.',
 'Outward Trade (Exports)': 'IFF: Outward Trade (Exports)',
 'Inward Trade (Imports)': 'IFF: Inward Trade (Imports)',
 'Top Flow': 'IFF: Top Flow',
 'Top Vulnerability': 'IFF: Top Vulnerability',
 'Vulnerability Region': 'IFF: Vulnerability Region',
 'Top1': 'IFF: Top1',
 'Top2': 'IFF: Top2',
 'Top3': 'IFF: Top3',
 'fsi_2022_rank': 'FSI_Rank',
 #'FSI2022_Share': 'FSI_Share',
 'fsi_2022_score': 'FSI_Score',
 'cthi_2021_rank': 'CTHI_Rank',
 'cthi_2021_share': 'CTHI_Share',
 'cthi_2021_score': 'CTHI_Score',
 #'Children lives lost due to tax revenue loss (total)': 'Children lives lost due to tax revenue loss (total)',
 'govt_exp_educ_gdp': 'WBD: Government education expenditure',
 'who_gvt_health_expenditure': 'WHO: Government health expenditure',
 'total_taxes_revenue': 'GRD: Total tax revenue',
 'cit_revenue': 'GRD: Total corporate income revenue',
 'gdp': 'GDP',
 'population': 'POP',
 'region_tjn': 'Region',
 'income_class': 'Income Class',
 'ukt': 'UK territory',
 'month_wage': 'Average wage'})

In [104]:
# Rename columns, drops rows with missing values in specific columns, excludes a particular ISO-3 code, creates a "Country" column using ISO-3 code mapping, sets a specific region for an ISO-3 code, creates an "IncomeClass2" column based on "Income Class", and displays the resulting DataFrame "df_merged"
df_merged = df_merged.rename(columns = rename_cols)
df_merged = df_merged.dropna(subset=["OW: Tax base loss (USD million)","Harm OW: Total (USD million)","TA: Tax base loss (USD million)","TA: Tax base gain (USD million)"],how="all")
df_merged = df_merged.loc[df_merged["ISO-3 code of country"]!="ATA"]
df_merged["Country"] = df_merged["ISO-3 code of country"].map(iso3_to_name)
df_merged.loc[df_merged["ISO-3 code of country"]=="PUS","Region"] = 'Caribean/American isl.'
df_merged["Income Class"] = df_merged["Income Class"].str.contains("Low").replace({False: "Higher", True: "Lower"})
df_merged.head()

,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,TA: Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,TA: Number countries reporting,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,TA: Tax base gain (USD million),TA: Tax base loss (USD million),TA: Min. Tax base gain (USD million),TA: Min. Tax base loss (USD million),TA: Max. Tax base gain (USD million),TA: Max. Tax base loss (USD million),TA: Tax revenue gain using CIT (USD million),TA: Tax revenue gain using ETR (USD million),TA: Tax revenue loss using CIT (USD million),TA: Tax revenue loss using ETR (USD million),TA: Min. Tax revenue gain using CIT (USD million),TA: Min. Tax revenue gain using ETR (USD million),TA: Min. Tax revenue loss using CIT (USD million),TA: Min. Tax revenue loss using ETR (USD million),TA: Max. Tax revenue gain using CIT (USD million),TA: Max. Tax revenue gain using ETR (USD million),TA: Max. Tax revenue loss using CIT (USD million),TA: Max. Tax revenue loss using ETR (USD million),TA: Robust,iso3,GRD: Total tax revenue,GRD: Total corporate income revenue,WBD: Government education expenditure,POP,Average wage,WHO: Government health expenditure,GDP,CTHI_Rank,CTHI_Share,CTHI_Score,FSI_Rank,FSI_Score,Region,eu28,oecd,g20,UK territory,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million),Income Class,Country
0,0.209388,104089.453125,0.097129,38513.097656,0.121643,20859.845703,USA,474005.0,469128.0,479168.0,1.0,0.227980,43832.0,43198.0,44497.0,12014.1900,10145.3160,2.0,27.0,0.27000,0.0,0.0,0.0,0.00,0.0000,1.0,129375.3600,109250.3040,-0.0,517837.0,-0.0,512326.0,-0.0,523665.0,-0.000000,-0.000000,139815.990000,118056.222832,-0.000000,-0.000000,138328.020000,116799.827781,-0.000000,-0.000000,141389.550000,119384.887386,2.0,USA,3.766917e+12,2.049875e+11,1.008652e+12,3.268382e+08,4400.00,1.747848e+12,2.053306e+13,25.0,0.011610,46.898267,1.0,67.425,Northern America,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.080189,Higher,United States
1,0.122834,61062.438965,0.431413,27478.097656,0.178115,30543.751953,GBR,70516.0,69887.0,71185.0,1.0,0.082819,17369.0,17044.0,17695.0,3362.0500,1465.1460,2.0,26.0,0.19000,-118088.0,-119520.0,-116726.0,-22177.94,-9664.9128,2.0,13525.1500,5894.1180,118088.0,87885.0,119520.0,86931.0,116726.0,88880.0,22436.719718,9779.973428,16698.149790,7278.580082,22708.799715,9898.570761,16516.889793,7199.570405,22177.939722,9667.173450,16887.199788,7360.985352,2.0,GBR,7.662662e+11,7.564826e+10,1.487426e+11,6.646034e+07,3216.68,2.261647e+11,2.878152e+12,13.0,0.031230,69.204656,13.0,47.175,Europe,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.013602,Higher,United Kingdom
2,0.055693,27685.537720,1.389173,13289.058594,0.052210,8953.128906,IRL,3435.0,3407.0,3465.0,1.0,0.115658,1469.0,1406.0,1538.0,192.2500,177.9466,2.0,19.0,0.12500,-40232.0,-42043.0,-38600.0,-4825.00,-4466.0200,2.0,433.1250,400.9005,40232.0,4904.0,42043.0,4813.0,38600.0,5003.0,5029.000000,4653.153367,613.000000,567.186919,5255.375000,4862.610037,601.625000,556.662039,4825.000000,4464.399482,625.375000,578.637062,2.0,IRL,7.161200e+10,1.225976e+10,1.308754e+10,4.867316e+06,3887.48,1.974858e+10,3.857367e+11,11.0,0.032960,77.079455,27.0,47.200,Europe,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Higher,Ireland
3,0.045214,22476.354980,0.116424,10114.359375,0.000000,0

In [105]:
#Impute missing values
# Calculate health expenditure as a percentage of GDP, education expenditure as a percentage of GDP, total tax revenue as a percentage of GDP, and corporate income revenue as a percentage of GDP, and assigns these values to the corresponding columns in the DataFrame "df_merged"
df_merged["health_gdp"] = df_merged["WHO: Government health expenditure"]/df_merged["GDP"]
df_merged["educ_gdp"] = df_merged["WBD: Government education expenditure"]/df_merged["GDP"]
df_merged["tax_rev_gdp"] = df_merged["GRD: Total tax revenue"]/df_merged["GDP"]
df_merged["c_tax_rev_gdp"] = df_merged["GRD: Total corporate income revenue"]/df_merged["GDP"]

# Calculate the average values of government health expenditure as a percentage of GDP, government education expenditure as a percentage of GDP, total tax revenue as a percentage of GDP, and corporate income revenue as a percentage of GDP, grouped by income class, and assigns these values to the corresponding columns in the DataFrame "reg_av"
reg_av = df_merged.groupby("Income Class").sum()
reg_av["health_gdp"] = reg_av["WHO: Government health expenditure"]/reg_av["GDP"]
reg_av["educ_gdp"] = reg_av["WBD: Government education expenditure"]/reg_av["GDP"]
reg_av["tax_rev_gdp"] = reg_av["GRD: Total tax revenue"]/reg_av["GDP"]
reg_av["c_tax_rev_gdp"] = reg_av["GRD: Total corporate income revenue"]/reg_av["GDP"]

# Convert it to dictionnary
reg_av = reg_av.to_dict()

# Assign the average values of the columns "health_gdp", "educ_gdp", "tax_rev_gdp", and "c_tax_rev_gdp" from the DataFrame "reg_av" to the corresponding NaN values in the same columns of the DataFrame "df_merged" based on the income class.
for v in ["health_gdp","educ_gdp","tax_rev_gdp","c_tax_rev_gdp"]:
    df_merged.loc[np.isnan(df_merged[v]),v] =  df_merged.loc[np.isnan(df_merged[v]),"Income Class"].map(reg_av[v])

# Impute the values for government health expenditure, government education expenditure, total tax revenue, and total corporate income revenue in "df_merged" by multiplying the GDP values with the corresponding ratios.
df_merged["WHO: Government health expenditure (imp)"] = df_merged["health_gdp"]*df_merged["GDP"]
df_merged["WBD: Government education expenditure (imp)"] = df_merged["educ_gdp"]*df_merged["GDP"]
df_merged["GRD: Total tax revenue (imp)"] = df_merged["tax_rev_gdp"]*df_merged["GDP"]
df_merged["GRD: Total corporate income revenue (imp)"] = df_merged["c_tax_rev_gdp"]*df_merged["GDP"]

In [106]:
#Tax losses
# Update the "df_merged" dataframe by adding columns for tax revenue losses using CIT and ETR, while also handling missing values and replacing negative values with 0 in the "Loss OW" column
df_merged["Loss OW (USD million)"] = df_merged['OW: Tax revenue loss (USD million)'].fillna(0)
df_merged.loc[df_merged["Loss OW (USD million)"]<0,"Loss OW (USD million)"] = 0
for st in ["","Min. ","Max. "]:
    df_merged[f"{st}Loss TA using CIT (USD million)"] = df_merged['TA: Tax revenue loss using CIT (USD million)'].fillna(0)
    df_merged[f"{st}Loss TA using ETR (USD million)"] = df_merged['TA: Tax revenue loss using ETR (USD million)'].fillna(0)

# Create a new column "GDP (only lossers)" in the "df_merged" dataframe and sets its values to the same as the "GDP" column, replacing any negative values with 0.
df_merged["GDP (only lossers)"] = df_merged["GDP"].copy()
df_merged.loc[df_merged["GDP (only lossers)"]<0,"GDP (only lossers)"] = 0

# Calculate various loss metrics for each tax type (CIT and ETR) in the "df_merged" dataframe, including total loss in USD million, percentage of GDP, percentage of government tax revenue, global and regional percentages of GDP and government tax revenue, per capita loss, percentages relative to government education and health expenditures, government corporate income revenue, and the number of nurses
for tax in ["CIT","ETR"]:
    df_merged[f"Loss Total using {tax} (USD million)"] = df_merged[f"Loss TA using {tax} (USD million)"].fillna(0) + df_merged["Loss OW (USD million)"].fillna(0)
    df_merged[f"Loss Total using {tax} (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GDP"]
    df_merged[f"Loss Total using {tax} (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} Global (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GDP"].sum()
    df_merged[f"Loss Total using {tax} Regional (% GDP)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GDP"].transform(sum)
    df_merged[f"Loss Total using {tax} Global (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GRD: Total tax revenue (imp)"].sum()
    df_merged[f"Loss Total using {tax} Regional (% gvt tax revenue)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GRD: Total tax revenue (imp)"].transform(sum)
    df_merged[f"Loss Total using {tax} (per capita)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["POP"]
    df_merged[f"Loss Total using {tax} (% Education)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged['WBD: Government education expenditure']
    df_merged[f"Loss Total using {tax} (% Health)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["WHO: Government health expenditure"]
    df_merged[f"Loss Total using {tax} (%  gvt corporate revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total corporate income revenue"]
    df_merged[f"Loss Total using {tax} (%  gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} (# Nurses)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["Average wage"]/12


In [107]:
# Retrieve specific loss metrics related to CIT (Corporate Income Tax) in the USA, including the loss total as a percentage of government tax revenue, the loss total in USD million, the loss in tax revenue using CIT in USD million, and the loss in offshore wealth in USD million. These metrics help assess the impact of tax-related losses in the country.
df_merged.loc[df_merged["ISO-3 code of country"]=="USA",[f"Loss Total using CIT (% gvt tax revenue)","Loss Total using CIT (USD million)","Loss TA using CIT (USD million)","Loss OW (USD million)"]]

,Loss Total using CIT (% gvt tax revenue),Loss Total using CIT (USD million),Loss TA using CIT (USD million),Loss OW (USD million)
0,4.734085,178329.087656,139815.99,38513.097656


In [108]:
df_merged["TA: Tax revenue loss using ETR (USD million)"].sum()
print("TRL {0:2,.0f}B (95% CI {1:2,.0f}-{2:2,.0f}B)".format(*1e-3*df_merged[["TA: Tax revenue loss using CIT (USD million)","TA: Min. Tax revenue loss using CIT (USD million)","TA: Max. Tax revenue loss using CIT (USD million)"]].sum()))

TRL 312B (95% CI 305-319B)


In [109]:
#Harm to others
#OW: Already in the file
df_merged["Harm OW: Total (% total)"] *= 100

# Calculate various columns in the DataFrame df_merged related to tax revenue loss using different methods (CIT and ETR) and percentage totals, based on given tax base gain and loss values
for st in ["","Min. ","Max. "]:
    #Total tax lost
    #OW already in the file
    df_merged[f"{st}Base gain TA (USD million)"] = df_merged[f'TA: {st}Tax base gain (USD million)'].fillna(0)
    df_merged[f"{st}Harm TA: Total (% total)"] = 100*df_merged[f"{st}Base gain TA (USD million)"]/df_merged[f"{st}Base gain TA (USD million)"].sum()
    
    total_loss_ta_cit = df_merged[f"{st}Loss TA using CIT (USD million)"].sum()
    total_loss_ta_etr = df_merged[f"{st}Loss TA using ETR (USD million)"].sum()
    df_merged[f"{st}Harm TA: Total using CIT (USD million)"] = total_loss_ta_cit*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100
    df_merged[f"{st}Harm TA: Total using ETR (USD million)"] = total_loss_ta_etr*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100

# Calculate various columns in the DataFrame df_merged related to total harm using different tax methods (CIT and ETR), including total harm values in USD million and percentage of the total, as well as the number of nurses affected based on the total harm percentage.
for tax in ["CIT","ETR"]:
    total_loss = df_merged[f"Loss Total using {tax} (USD million)"].sum()
    df_merged[f"Harm: Total using {tax} (USD million)"] = df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]
    df_merged[f"Harm: Total using {tax} (% total)"] = 100*df_merged[f"Harm: Total using {tax} (USD million)"]/(df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]).sum()

    total_nurses = df_merged[f'Loss Total using {tax} (# Nurses)'].sum()
    df_merged[f"Harm: Total using {tax} (# Nurses)"] = total_nurses*df_merged[f"Harm: Total using {tax} (% total)"]/100

df_merged["Year"] = 2021

#Add missing values
# Set the values of specific columns in the DataFrame df_merged (including "OW: Tax base loss (USD million)", "OW: Tax revenue loss (USD million)", "Harm OW: Total (% total)", "Harm OW: Total (USD million)", and "Loss OW (USD million)") to NaN if the corresponding values in the column "OW: Tax base loss (USD million)" are NaN.
df_merged.loc[np.isnan(df_merged["OW: Tax base loss (USD million)"]),
              ['OW: Tax base loss (USD million)',
 'OW: Tax revenue loss (USD million)',
 'Harm OW: Total (% total)',
 'Harm OW: Total (USD million)','Loss OW (USD million)']] = np.NaN

# Set specific columns in the DataFrame df_merged to NaN if both "TA: Tax base loss (USD million)" and "OW: Tax base loss (USD million)" have NaN values
cond = np.isnan(df_merged["TA: Tax base loss (USD million)"]) & np.isnan(df_merged["OW: Tax base loss (USD million)"])
df_merged.loc[cond,['Loss Total (USD million)',
 'Loss Total (% GDP)',
 'Loss Total (% gvt tax revenue)',
 'Loss Total Global (% GDP)',
 'Loss Total Regional (% GDP)',
 'Loss Total Global (% gvt tax revenue)',
 'Loss Total Regional (% gvt tax revenue)',
 'Loss Total (per capita)',
 'Loss Total (% Education)',
 'Loss Total (% Health)',
 'Loss Total (%  gvt corporate revenue)',
 'Loss Total (%  gvt tax revenue)',
 'Loss Total (# Nurses)']] = np.nan

In [110]:
# Perform an outer merge of the DataFrame df_merged with another DataFrame named other_info
df_merged = pd.merge(df_merged,other_info,how="outer")
df_merged.loc[df_merged["Country"]=="India"]

,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,TA: Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,TA: Number countries reporting,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,TA: Tax base gain (USD million),TA: Tax base loss (USD million),TA: Min. Tax base gain (USD million),TA: Min. Tax base loss (USD million),TA: Max. Tax base gain (USD million),TA: Max. Tax base loss (USD million),TA: Tax revenue gain using CIT (USD million),TA: Tax revenue gain using ETR (USD million),TA: Tax revenue loss using CIT (USD million),TA: Tax revenue loss using ETR (USD million),TA: Min. Tax revenue gain using CIT (USD million),TA: Min. Tax revenue gain using ETR (USD million),TA: Min. Tax revenue loss using CIT (USD million),TA: Min. Tax revenue loss using ETR (USD million),TA: Max. Tax revenue gain using CIT (USD million),TA: Max. Tax revenue gain using ETR (USD million),TA: Max. Tax revenue loss using CIT (USD million),TA: Max. Tax revenue loss using ETR (USD million),TA: Robust,iso3,GRD: Total tax revenue,GRD: Total corporate income revenue,...,Loss Total using ETR (% gvt corporate revenue),Loss Total using ETR (% gvt tax revenue),Loss Total using ETR (# Nurses),Base gain TA (USD million),Harm TA: Total (% total),Harm TA: Total using CIT (USD million),Harm TA: Total using ETR (USD million),Min. Base gain TA (USD million),Min. Harm TA: Total (% total),Min. Harm TA: Total using CIT (USD million),Min. Harm TA: Total using ETR (USD million),Max. Base gain TA (USD million),Max. Harm TA: Total (% total),Max. Harm TA: Total using CIT (USD million),Max. Harm TA: Total using ETR (USD million),Harm: Total using CIT (USD million),Harm: Total using CIT (% total),Harm: Total using CIT (# Nurses),Harm: Total using ETR (USD million),Harm: Total using ETR (% total),Harm: Total using ETR (# Nurses),Year,Loss Total (USD million),Loss Total (% GDP),Loss Total (% gvt tax revenue),Loss Total Global (% GDP),Loss Total Regional (% GDP),Loss Total Global (% gvt tax revenue),Loss Total Regional (% gvt tax revenue),Loss Total (per capita),Loss Total (% Education),Loss Total (% Health),Loss Total (% gvt corporate revenue),Loss Total (% gvt tax revenue),Loss Total (# Nurses),total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,ukt,income_class
48,0.001017,505.768108,0.003524,181.469604,0.0,0.0,IND,34685.0,34330.0,35049.0,1.0,0.413186,30553.0,29268.0,31962.0,15444.0384,13206.6984,2.0,22.0,0.48316,0.0,0.0,0.0,0.0,0.0,1.0,16935.6768,14482.2468,-0.0,65238.0,-0.0,63598.0,-0.0,67011.0,-0.0,-0.0,31520.393314,26955.408947,-0.0,-0.0,30728.010883,26277.784393,-0.0,-0.0,32377.036028,27687.9872,2.0,IND,NaN,NaN,...,NaN,NaN,9.625054e+06,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,2021.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.179488e+11,1.369003e+09,234.95,2.582103e+10,2.702930e+12,NaN,NaN,NaN,36.0,54.725,Asia,0.0,Lower-middle income


In [111]:
def robust(s):
    return a

# Iterate over each row in the DataFrame df_merged and assigns a background color based on the value of the "TA: Robust" column, storing the colors in a list called a.
a = []
for i,row in df_merged.iterrows():
    if row["TA: Robust"]==2:
        a.append("background-color: #44b0c6")
    elif row["TA: Robust"]==1:
        a.append("background-color: #94c9d4")
    else:
        a.append("background-color: white")
        
    
# df_merged.style.apply(robust)

In [112]:
# Applies the "robust" style to the DataFrame df_merged and saves it as an Excel file in different locations specified by the file paths provided.
df_merged.style.apply(robust).to_excel("~/Downloads/combined_output.xlsx",index=None)
df_merged.style.apply(robust).to_excel(final_table_output,index=None)
df_merged.style.apply(robust).to_excel(final_table_output_workstream,index=None)

In [113]:
# Select the rows in the DataFrame df_merged where the value in the "Income Class" column, after filling missing values with "X", is equal to "X".
df_merged.loc[df_merged["Income Class"].fillna("X")=="X"]

,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,TA: Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,TA: Number countries reporting,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,TA: Tax base gain (USD million),TA: Tax base loss (USD million),TA: Min. Tax base gain (USD million),TA: Min. Tax base loss (USD million),TA: Max. Tax base gain (USD million),TA: Max. Tax base loss (USD million),TA: Tax revenue gain using CIT (USD million),TA: Tax revenue gain using ETR (USD million),TA: Tax revenue loss using CIT (USD million),TA: Tax revenue loss using ETR (USD million),TA: Min. Tax revenue gain using CIT (USD million),TA: Min. Tax revenue gain using ETR (USD million),TA: Min. Tax revenue loss using CIT (USD million),TA: Min. Tax revenue loss using ETR (USD million),TA: Max. Tax revenue gain using CIT (USD million),TA: Max. Tax revenue gain using ETR (USD million),TA: Max. Tax revenue loss using CIT (USD million),TA: Max. Tax revenue loss using ETR (USD million),TA: Robust,iso3,GRD: Total tax revenue,GRD: Total corporate income revenue,...,Loss Total using ETR (% gvt corporate revenue),Loss Total using ETR (% gvt tax revenue),Loss Total using ETR (# Nurses),Base gain TA (USD million),Harm TA: Total (% total),Harm TA: Total using CIT (USD million),Harm TA: Total using ETR (USD million),Min. Base gain TA (USD million),Min. Harm TA: Total (% total),Min. Harm TA: Total using CIT (USD million),Min. Harm TA: Total using ETR (USD million),Max. Base gain TA (USD million),Max. Harm TA: Total (% total),Max. Harm TA: Total using CIT (USD million),Max. Harm TA: Total using ETR (USD million),Harm: Total using CIT (USD million),Harm: Total using CIT (% total),Harm: Total using CIT (# Nurses),Harm: Total using ETR (USD million),Harm: Total using ETR (% total),Harm: Total using ETR (# Nurses),Year,Loss Total (USD million),Loss Total (% GDP),Loss Total (% gvt tax revenue),Loss Total Global (% GDP),Loss Total Regional (% GDP),Loss Total Global (% gvt tax revenue),Loss Total Regional (% gvt tax revenue),Loss Total (per capita),Loss Total (% Education),Loss Total (% Health),Loss Total (% gvt corporate revenue),Loss Total (% gvt tax revenue),Loss Total (# Nurses),total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,ukt,income_class
11,1.547569e-02,7693.150330,0.251661,3077.260254,0.000000,0.000000,TWN,0.0,0.0,0.0,NaN,0.160292,31616.0,30712.0,32680.0,6536.0,5238.6040,2.0,18.0,0.2,0.0,0.0,0.0,0.0,0.0000,1.0,NaN,NaN,-0.0,31616.0,-0.0,30712.0,-0.0,32680.0,-0.0,-0.000000,6323.2,5067.784975,-0.0,-0.000000,6142.4,4922.881204,-0.0,-0.000000,6536.0,5238.335431,NaN,TWN,NaN,NaN,...,NaN,NaN,NaN,-0.0,-0.000000,-0.000000,-0.000000,-0.0,-0.000000,-0.000000,-0.000000,-0.0,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2021.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.113910e+11,55.0,0.002220,43.482856,17.0,60.125,Asia,0.0,NaN
18,1.462991e-02,7272.703552,40.997574,1454.540771,0.000000,0.000000,JEY,0.0,0.0,0.0,NaN,0.007979,1432.0,1225.0,1837.0,0.0,14.6960,2.0,11.0,0.0,-18727.0,-23993.0,-16067.0,-0.0,-128.5360,2.0,NaN,NaN,18727.0,1432.0,23993.0,1225.0,16067.0,1837.0,0.0,149.420903,0.0,11.425788,0.0,191.437802,0.0,9.774155,0.

In [114]:
# ## Work for ATP (danish people)
# #Main results
# final_data_output = f"{path_files_temp}{YEAR_CBCR}_replicates.csv"
# atp_prbook = pd.read_csv(final_data_output,sep="\t").reset_index(drop=True)
# atp_prbook = atp_prbook.dropna()
# atp_prbook = atp_prbook.loc[atp_prbook["profits"]>0]
# atp_prbook = atp_prbook.groupby(["iso3_d","n_rep"]).sum()[["profits"]].groupby("iso3_d").median().reset_index()
# atp_prbook.head()



In [115]:
# atp_other = pd.read_excel(final_table_output_workstream)
# atp_other["Gain - Loss"] = atp_other["TA: Tax base gain (USD million)"] -  atp_other["TA: Tax base loss (USD million)"]
# atp_other = atp_other[['ISO-3 code of country','ETR', "Gain - Loss"]]
# atp_other.columns = ["iso3_d","ETR","Gain - Loss"]
# atp_other = atp_other.loc[atp_other["Gain - Loss"]>0]
# atp_other = pd.merge(atp_other,atp_prbook,how="left")
# atp_other["profits"] /= 1E6

In [116]:
# atp_other["var"] = 100*atp_other["Gain - Loss"]/atp_other["profits"] * atp_other["Gain - Loss"]/atp_other["Gain - Loss"].sum()
# atp_other=  atp_other.sort_values(by="var",ascending=False)
# atp_other.to_excel("C:/Users/javga/Downloads/temp.xlsx")

In [117]:
# atp_other.loc[atp_other["profits"]<atp_other["Gain - Loss"]]